This notebook decomposes `HeatingSystem` into its component parts using the same four structural constructs introduced in Chapter 1; after running it you can see the same abstract-def, part-def, specialization, and composition pattern applied one level down.

Chapter 1 built the toaster's top-level structure: an abstract base, concrete types, specialization, and composition. Chapter 6 applies those four constructs to decompose `HeatingSystem` into a `ResistanceCoil` and a `PowerWire`, both specializations of `HeatingElement`.

A `HeatingAssembly` part definition composes them — it specializes `HeatingSystem` and owns both subparts. `model.query()` can then return the full set of part definitions at this level.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    requirement def HeatingReq {
        subject heater : Heater;
        require constraint { heater.power >= 600.0 }
    }
    requirement heating : HeatingReq;
    part efficient : Heater;
    part weak : Heater { attribute :>> power = 400.0; }
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement {
        attribute resistance : Real default = 12.0;
    }
    part def PowerWire :> HeatingElement {
        attribute gauge : Real default = 14.0;
    }
    part def HeatingAssembly :> HeatingSystem {
        part coil : ResistanceCoil;
        part wire : PowerWire;
    }
    part heatingEvidence {
        assert satisfy heating by efficient;
        assert satisfy heating by weak;
    }
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")

In [ ]:
# Negative control: composing a part typed by an undefined type raises "unresolved reference".
# Composition requires the type to be declared — the same rule applies at every level.
bad_source = """
package BadSecond {
    private import ScalarValues::*;
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement;
    part def HeatingAssembly {
        part coil : ResistanceCoil;
        part wire : UndefinedType;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print(f"Neg control diagnostics: {bad.diagnostics[0].message!r}")

In [ ]:
# List all PartDefinition elements — should include the second-level types
part_defs = [e.as_dict() for e in model.query()
             if e.as_dict().get("@type") == "PartDefinition"]
for pd in part_defs:
    name = pd.get("declaredName") or pd.get("name", "?")
    abstract = pd.get("isAbstract") == "true"
    print(f"  {'abstract ' if abstract else ''}part def {name}")

The four structural constructs from Chapter 1 (A-F) are applied one level down in the model hierarchy and parsed by OpenSysML (O-S); `model.query()` returns all `PartDefinition` elements including the second-level `HeatingElement`, `ResistanceCoil`, `PowerWire`, and `HeatingAssembly` (E).

Try the chapter exercise in `exercises/ch06/exercise.ipynb`: decompose `BrewUnit` into an `Impeller` and a `FilterBasket`, both specializations of a `BrewComponent` abstract part, and confirm all three appear in the `model.query()` result.